# Capítulo 13: Optimización de Operaciones y Logística

> *«La logística es como una orquesta sin director: cada músico toca su instrumento perfectamente, pero si nadie coordina los tiempos, el resultado es ruido, no sinfonía.»*

Este notebook acompaña al Capítulo 13 del libro *Ciencia de Datos sin Filtros*. Aquí implementaremos análisis de operaciones logísticas, predicción de retrasos y optimización de rutas usando un dataset de 25,000 entregas de última milla.

**Dataset:** Entregas de logística de última milla (25,000 registros, 5 repartidores, 4 regiones)

**Contenido:**
1. Carga y diagnóstico de calidad de datos
2. Análisis exploratorio de retrasos
3. Predicción de retrasos con Random Forest
4. Optimización de rutas simulada
5. Análisis ético de rendimiento de trabajadores
6. Prompts para IA en logística
7. Framework de verificación

---
## Celda 1: Importaciones

Importamos las librerías necesarias para análisis logístico y machine learning.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerías cargadas correctamente')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

---
## Celda 2: Carga de Datos de Entregas

Cargamos el dataset de 25,000 entregas de última milla.

**Columnas clave:**
- `delivery_partner`: Repartidor (Partner-A a Partner-E)
- `distance_km`: Distancia en kilómetros
- `delivery_time_hours`: Tiempo real de entrega
- `expected_time_hours`: Tiempo esperado
- `delayed`: ¿Se retrasó? (Yes/No)
- `delivery_rating`: Calificación del cliente (1-5)

In [ ]:
df = pd.read_csv('../datos/datos_logistica_entregas.csv')

print(f'Dimensiones: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
print(f'Repartidores: {df["delivery_partner"].unique().tolist()}')
print(f'Regiones: {df["region"].unique().tolist()}')
print(f'Tipos de paquete: {df["package_type"].unique().tolist()}')
print()
df.head(10)

---
## Celda 3: Diagnóstico de Calidad de Datos

Los datos logísticos son notoriamente sucios. Detectamos anomalías, duplicados e inconsistencias.

In [ ]:
print('DIAGNÓSTICO DE CALIDAD DE DATOS')
print('=' * 60)

print('\n1. TIPOS DE DATOS:')
print(df.dtypes)

nulos = df.isnull().sum()
print('\n2. VALORES NULOS:')
print(nulos[nulos > 0] if nulos.sum() > 0 else '   No hay valores nulos')

duplicados = df.duplicated().sum()
print(f'\n3. DUPLICADOS: {duplicados} filas duplicadas')

print('\n4. ESTADÍSTICAS BÁSICAS:')
print(df.describe())

# Anomalías
negativos = df[df['delivery_time_hours'] < 0]
excesivos = df[df['delivery_time_hours'] > 20]
sospechosos = df[(df['distance_km'] > 50) & (df['delivery_time_hours'] < 1)]

print('\n5. ANOMALÍAS:')
print(f'   Tiempos negativos: {len(negativos)} registros')
print(f'   Tiempos >20 horas: {len(excesivos)} registros')
print(f'   Distancia >50km en <1 hora: {len(sospechosos)} registros')

---
## Celda 4: Análisis de Retrasos por Repartidor

Analizamos las tasas de retraso y su impacto en satisfacción del cliente.

In [ ]:
# Tasa de retraso por repartidor
retraso_por_partner = df.groupby('delivery_partner').agg({
    'delivery_id': 'count',
    'delayed': lambda x: (x == 'Yes').sum(),
    'delivery_time_hours': 'mean',
    'delivery_rating': 'mean',
    'delivery_cost': 'mean'
}).round(2)

retraso_por_partner.columns = ['total_envios', 'retrasados', 'tiempo_promedio', 'rating_promedio', 'costo_promedio']
retraso_por_partner['tasa_retraso'] = ((retraso_por_partner['retrasados'] / retraso_por_partner['total_envios']) * 100).round(1)
retraso_por_partner = retraso_por_partner.sort_values('tasa_retraso', ascending=False)

print('TASA DE RETRASO POR REPARTIDOR')
print('=' * 70)
print(retraso_por_partner)

# Impacto del clima
print('\n' + '=' * 60)
retraso_por_clima = df.groupby('weather_condition').agg({
    'delivery_id': 'count',
    'delayed': lambda x: (x == 'Yes').sum(),
    'delivery_time_hours': 'mean',
    'delivery_rating': 'mean'
}).round(2)
retraso_por_clima.columns = ['total', 'retrasados', 'tiempo_promedio', 'rating_promedio']
retraso_por_clima['tasa_retraso'] = ((retraso_por_clima['retrasados'] / retraso_por_clima['total']) * 100).round(1)
print('IMPACTO DEL CLIMA EN RETRASOS')
print(retraso_por_clima)

---
## Celda 5: Predicción de Retrasos con Random Forest

Entrenamos un modelo de clasificación para predecir qué envíos se retrasarán.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()

# Encode variables categóricas
le_dict = {}
categorical_cols = ['delivery_partner', 'package_type', 'vehicle_type',
                    'delivery_mode', 'region', 'weather_condition']

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col + '_encoded'] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le

# Target
df_model['delayed_binary'] = (df_model['delayed'] == 'Yes').astype(int)

# Features
feature_cols = [col + '_encoded' for col in categorical_cols] + \
               ['distance_km', 'package_weight_kg', 'expected_time_hours']

X = df_model[feature_cols]
y = df_model['delayed_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test: {X_test.shape[0]} muestras')
print(f'Tasa de retraso train: {y_train.mean():.2%}')
print(f'Tasa de retraso test: {y_test.mean():.2%}')

# Entrenar modelo
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('\nREPORTE DE CLASIFICACIÓN - PREDICCIÓN DE RETRASOS')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['A tiempo', 'Retrasado']))

---
## Celda 6: Importancia de Features

Identificamos qué variables son más importantes para predecir retrasos.

In [ ]:
importancia = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('IMPORTANCIA DE FEATURES')
print('=' * 50)
for _, row in importancia.iterrows():
    barra = '█' * int(row['importance'] * 100)
    print(f"{row['feature']:30s} | {row['importance']:.3f} {barra}")

print('\n' + '=' * 60)
print('INTERPRETACIÓN:')
print('Las features con mayor importancia son las que más influyen')
print('en la predicción de retrasos. Esto no implica causalidad.')

---
## Celda 7: Simulación de Optimización de Rutas

Simulamos la diferencia entre rutas aleatorias y optimizadas para estimar ahorros potenciales.

In [ ]:
np.random.seed(42)

def simular_rutas(df, n_rutas=50):
    """Simula la diferencia entre rutas aleatorias y optimizadas."""
    resultados = []
    for i in range(n_rutas):
        entregas = df.sample(10)
        distancia_total_aleatoria = entregas['distance_km'].sum()
        factor_optimizacion = np.random.uniform(0.75, 0.85)
        distancia_total_optimizada = distancia_total_aleatoria * factor_optimizacion
        tiempo_ahorrado = (distancia_total_aleatoria - distancia_total_optimizada) * 0.12
        costo_ahorrado = (distancia_total_aleatoria - distancia_total_optimizada) * 0.3
        resultados.append({
            'ruta': f'Ruta-{i+1}',
            'distancia_aleatoria': round(distancia_total_aleatoria, 1),
            'distancia_optimizada': round(distancia_total_optimizada, 1),
            'reduccion_pct': round((1 - factor_optimizacion) * 100, 1),
            'tiempo_ahorrado_h': round(tiempo_ahorrado, 2),
            'costo_ahorrado_usd': round(costo_ahorrado, 2)
        })
    return pd.DataFrame(resultados)

rutas_df = simular_rutas(df)

print('SIMULACIÓN: RUTAS ALEATORIAS vs OPTIMIZADAS')
print('=' * 70)
print(f"Promedio de reducción de distancia: {rutas_df['reduccion_pct'].mean():.1f}%")
print(f"Tiempo promedio ahorrado por ruta: {rutas_df['tiempo_ahorrado_h'].mean():.2f} horas")
print(f"Costo promedio ahorrado por ruta: ${rutas_df['costo_ahorrado_usd'].mean():.2f}")
n_rutas_anual = 25000 / 10
print(f'\nProyección anual ({int(n_rutas_anual)} rutas):')
print(f"  Tiempo total ahorrado: {rutas_df['tiempo_ahorrado_h'].mean() * n_rutas_anual:.0f} horas")
print(f"  Costo total ahorrado: ${rutas_df['costo_ahorrado_usd'].mean() * n_rutas_anual:,.2f}")

---
## Celda 8: Análisis Ético de Rendimiento

Analizamos si hay patrones de asignación sesgada y evaluamos el impacto ético de las optimizaciones.

In [ ]:
print('ANÁLISIS ÉTICO DE RENDIMIENTO POR REPARTIDOR')
print('=' * 70)

for partner in df['delivery_partner'].unique():
    datos = df[df['delivery_partner'] == partner]
    tasa_retraso = (datos['delayed'] == 'Yes').mean() * 100
    rating_prom = datos['delivery_rating'].mean()
    costo_prom = datos['delivery_cost'].mean()
    tiempo_prom = datos['delivery_time_hours'].mean()
    
    print(f'\n{partner}:')
    print(f'  Envíos: {len(datos):,}')
    print(f'  Tasa de retraso: {tasa_retraso:.1f}%')
    print(f'  Rating promedio: {rating_prom:.2f}')
    print(f'  Costo promedio: ${costo_prom:.2f}')
    print(f'  Tiempo promedio: {tiempo_prom:.2f}h')
    
    if tasa_retraso > 40:
        print(f'  ⚠️ ALERTA: Tasa de retraso > 40%. ¿Está recibiendo demasiadas entregas?')
    if rating_prom < 3.0:
        print(f'  ⚠️ ALERTA: Rating bajo. ¿Está el repartidor sobrecargado?')

# Distribución de paquetes
print('\n' + '=' * 60)
print('DISTRIBUCIÓN DE TIPOS DE PAQUETE POR REPARTIDOR')
print('=' * 60)
distribucion = pd.crosstab(df['delivery_partner'], df['package_type'], normalize='index') * 100
print(distribucion.round(1))

print('\nPREGUNTA ÉTICA:')
print('¿Estamos asignando paquetes pesados a los mismos repartidores todo el tiempo?')
print('¿Eso es eficiencia o explotación?')

---
## Celda 9: Prompts para IA en Logística

Generamos prompts específicos para que una IA nos ayude a optimizar operaciones.

In [ ]:
# Preparar datos para prompt
resumen_para_prompt = df.groupby('delivery_partner').agg({
    'delivery_id': 'count',
    'delayed': lambda x: (x == 'Yes').sum(),
    'delivery_time_hours': 'mean',
    'delivery_cost': 'mean',
    'delivery_rating': 'mean'
}).round(2)

print('DATOS RESUMEN PARA PROMPT DE IA')
print('=' * 60)
print(resumen_para_prompt)

prompt_ejemplo = """
DATOS DE LOGÍSTICA (25,000 entregas):
- Partner-A: {} envíos | {:.0f}% retrasos | Tiempo prom: {:.1f}h | Costo: ${:.2f}
- Partner-B: {} envíos | {:.0f}% retrasos | Tiempo prom: {:.1f}h | Costo: ${:.2f}
- Partner-C: {} envíos | {:.0f}% retrasos | Tiempo prom: {:.1f}h | Costo: ${:.2f}
- Partner-D: {} envíos | {:.0f}% retrasos | Tiempo prom: {:.1f}h | Costo: ${:.2f}
- Partner-E: {} envíos | {:.0f}% retrasos | Tiempo prom: {:.1f}h | Costo: ${:.2f}

CONTEXTO: Empresa de última milla en 4 regiones.

GENERAR:
1. Diagnóstico: ¿Por qué hay diferencias entre repartidores?
2. Estrategias: 5 acciones para reducir retrasos un 15%
3. Riesgos: ¿Qué podría salir mal con cada estrategia?
"""

# Llenar prompt con datos reales
partners_data = []
for partner in resumen_para_prompt.index:
    row = resumen_para_prompt.loc[partner]
    tasa = (row['retrasados'] / row['delivery_id']) * 100
    partners_data.append((
        int(row['delivery_id']), tasa, row['delivery_time_hours'], row['delivery_cost']
    ))

print('\n' + '=' * 60)
print('PROMPT GENERADO PARA IA')
print('=' * 60)
print(prompt_ejemplo.format(
    partners_data[0][0], partners_data[0][1], partners_data[0][2], partners_data[0][3],
    partners_data[1][0], partners_data[1][1], partners_data[1][2], partners_data[1][3],
    partners_data[2][0], partners_data[2][1], partners_data[2][2], partners_data[2][3],
    partners_data[3][0], partners_data[3][1], partners_data[3][2], partners_data[3][3],
    partners_data[4][0], partners_data[4][1], partners_data[4][2], partners_data[4][3]
))

---
## Celda 10: Verificación de Recomendaciones de IA

Evaluamos si las recomendaciones de IA son confiables y éticas.

In [ ]:
print('EJERCICIO: VERIFICACIÓN DE OPTIMIZACIÓN')
print('=' * 60)

print('\nRecomendación hipotética de IA:')
print("'Reasigna 10% de las entregas de Partner-A a Partner-E para reducir retrasos.'")

print('\nVerificaciones necesarias:')
print('1. ¿Partner-E tiene capacidad para absorber 10% más entregas?')
print('   → Revisar: ¿Cuántas entregas ya maneja por día?')
print('2. ¿El mejor rating de Partner-E se debe a mejor desempeño')
print('   o a que recibe menos entregas difíciles?')
print('3. ¿Qué pasa con la moral de Partner-A?')
print('   → Revisar: ¿Aumenta la rotación? ¿Cuesta reemplazarlo?')
print('4. ¿El cambio es sostenible o es un parche temporal?')
print('5. ¿Los datos de rating son confiables o están sesgados?')

# Verificación práctica
print('\n' + '=' * 60)
print('VERIFICACIÓN CON DATOS REALES:')
for partner in df['delivery_partner'].unique():
    datos = df[df['delivery_partner'] == partner]
    envios_por_dia = len(datos) / 30  # Asumiendo 30 días de datos
    print(f"{partner}: {envios_por_dia:.0f} envíos/día promedio")

print('\n' + '=' * 60)
print('LECCIÓN: Siempre verifica con datos antes de implementar')
print('una recomendación de IA. Los humanos tienen contexto que')
print('los datos no capturan.')